[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ItunuAjiboye/Tutorial_Host_Pathogen_Protein_Protein_Interaction_Prediction/blob/main/notebooks/03_Feature_Extraction.ipynb)
> This notebook is designed to be run in Google Colab. Click on "Open in Colab" to continue.

# Data Preprocessing

## Overview

This notebook prepares the host–pathogen protein interaction data for machine learning model development. The quality and structure of the input data are important for building reliable predictive models, as sequence errors, redundant proteins, inappropriate negative sampling, and information leakage between training and test datasets can lead to biased or overly optimistic performance estimates.

The preprocessing workflow is organised into five modules. Each module performs a specific step in preparing the interaction dataset while preserving the protein and interaction identifiers required for downstream analysis.

### Module 1 — Sequence Validation


### Module 2 — Homology Clustering

### Module 3 — Balanced and Imbalanced Dataset Construction

### Module 4 — Homology-Aware Data Partitioning

### Module 5 — Host-Only Homology-Aware Partitioning

## Notebook Workflow

The five modules collectively transform the initial interaction dataset into quality-controlled, class-balanced or imbalanced, and homology-aware datasets that are ready for downstream feature extraction and machine learning. The resulting datasets retain the host protein ID, pathogen protein ID, pair ID, sequence information, class label, and relevant homology cluster information to support reproducible analysis and downstream validation.



#Module 1: Sequence Validation
This module performs quality control on the protein sequences used in the
interaction datasets. Protein pairs containing non-standard amino acids
or sequences shorter than the specified minimum length are removed.

The validation is applied to both positive and negative interaction data
across all bacterial species.

there are two functions:

The **is_valid_protein()** function performs quality control on individual protein sequences by checking for missing values, non-standard amino acids, and sequences shorter than the specified minimum length.


The **validate_interaction_pairs()** function applies this validation to both host and pathogen sequences for every interaction pair and retains only pairs in which both sequences pass all quality-control criteria.

In [ ]:
#Mounting Google Drive to access files
import os
from google.colab import drive
drive.mount ('/content/my_drive')


In [ ]:


import pandas as pd

VALID_AAS = set("ACDEFGHIKLMNPQRSTVWY")


def is_valid_protein(seq, min_length=35):
    """Checks a single sequence for standard amino acids and minimum length."""
    if pd.isna(seq) or not isinstance(seq, str):
        return False                    #Check for missing or invalid data
    seq = seq.upper().strip()
    if len(seq) < min_length:
        return False
    return all(aa in VALID_AAS for aa in seq)


def validate_interaction_pairs(
    input_csv,
    species_label,
    host_seq_col="host_sequence",
    pathogen_seq_col="pathogen_sequence",
    min_length=35,
    host_id_col=None,
    pathogen_id_col=None,
):

    df = pd.read_csv(input_csv)

    df["host_sequence"] = df[host_seq_col].str.upper().str.strip()
    df["pathogen_sequence"] = df[pathogen_seq_col].str.upper().str.strip()

    valid_host = df["host_sequence"].apply(lambda s: is_valid_protein(s, min_length))
    valid_pathogen = df["pathogen_sequence"].apply(lambda s: is_valid_protein(s, min_length))

    df_valid = df[valid_host & valid_pathogen].copy()
    df_valid["pathogen_specie"] = species_label

    keep_cols = ["host_sequence", "pathogen_sequence", "pathogen_specie"]
    if host_id_col:
        keep_cols.insert(0, host_id_col)
        df_valid = df_valid.rename(columns={host_id_col: "host_protein_id"})
        keep_cols[0] = "host_protein_id"
    if pathogen_id_col:
        keep_cols.insert(-1, pathogen_id_col)
        df_valid = df_valid.rename(columns={pathogen_id_col: "pathogen_protein_id"})
        keep_cols[-2] = "pathogen_protein_id"

    df_valid = df_valid[[c for c in keep_cols if c in df_valid.columns]]

    print(f"[{species_label}] Validated {len(df_valid)} / {len(df)} pairs "
          f"({len(df) - len(df_valid)} removed)")
    return df_valid

##Run validation for all species, positive and negative

In [ ]:
file_path = '/content/my_drive/MyDrive/HPI'

species_files = {
    "bacillus":    {"positive": "bacillus_positive_pairs.csv",    "negative": "bacillus_negative_pool.csv"},
    "ecoli":       {"positive": "ecoli_positive_pairs.csv",       "negative": "ecoli_negative_pool.csv"},
    "francisella": {"positive": "francisella_positive_pairs.csv", "negative": "francisella_negative_pool.csv"},
    "yersinia":    {"positive": "yersinia_positive_pairs.csv",    "negative": "yersinia_negative_pool.csv"},
}

validated_positive = {}
validated_negative = {}

for species, files in species_files.items():
    validated_positive[species] = validate_interaction_pairs(
        f"{file_path}/{files['positive']}", species_label=species,
    )
    validated_positive[species].to_csv(f"{file_path}/validated_positive_{species}.csv", index=False)

    validated_negative[species] = validate_interaction_pairs(
        f"{file_path}/{files['negative']}", species_label=species,
    )
    validated_negative[species].to_csv(f"{file_path}/validated_negative_{species}.csv", index=False)

# Combine all positives into one master file (used at every imbalance ratio)
all_positive = pd.concat(validated_positive.values(), ignore_index=True)
all_positive.to_csv(f"{file_path}/AllPositiveHPI.csv", index=False)
print(f"Total validated positive pairs: {len(all_positive)}")

#Module 2: Homology Clustering
This module performs sequence-based homology clustering of all host and pathogen proteins used in the negative and positive interaction data. The clustering is performed using CD-HIT at a specified sequence identity threshold (0.4) and is used to identify groups of homologous proteins.

**Why Homology Clustering?**

Protein sequences that are highly similar may provide very similar information to a machine-learning model. If homologous proteins are distributed across training and testing datasets, the model may appear to perform well simply because it has encountered a highly similar protein during training.

Homology-aware partitioning helps reduce this potential source of information leakage by identifying groups of related proteins that can subsequently be kept within the same data partition or cross-validation fold.

In this workflow, CD-HIT is used to cluster protein sequences according to sequence similarity. The resulting cluster assignments provide the information required for downstream homology-aware data splitting.

**Workflow**

All validated positive interactions and negative pools are first combined to create the complete sequence universe:


    Positive data + Negative pools
              ↓
     Identify Unique protein sequences
              ↓
          CD-HIT clustering
              ↓
     Cluster assignments

In [ ]:

import os
import json
import subprocess
import pandas as pd


def write_fasta(sequences, filename, prefix):
    with open(filename, "w") as f:
        for i, seq in enumerate(sequences):
            f.write(f">{prefix}_{i}\n{seq}\n")


def build_id_maps(df, host_col="host_sequence", pathogen_col="pathogen_sequence"):
    host_sequences = df[host_col].dropna().unique()
    pathogen_sequences = df[pathogen_col].dropna().unique()

    host_map_df = pd.DataFrame({
        "host_protein_id": [f"HOST_{i}" for i in range(len(host_sequences))],
        "host_sequence": host_sequences,
    })
    pathogen_map_df = pd.DataFrame({
        "pathogen_protein_id": [f"PATHOGEN_{i}" for i in range(len(pathogen_sequences))],
        "pathogen_sequence": pathogen_sequences,
    })
    return host_map_df, pathogen_map_df, host_sequences, pathogen_sequences


def run_cdhit(input_fasta, output_prefix, identity=0.4, word_size=2, memory_mb=16000, threads=8):
    cmd = [
        "cd-hit", "-i", input_fasta, "-o", output_prefix,
        "-c", str(identity), "-n", str(word_size),
        "-d", "0", "-M", str(memory_mb), "-T", str(threads),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-500:])
    if result.returncode != 0:
        raise RuntimeError(f"CD-HIT failed: {result.stderr}")
    return f"{output_prefix}.clstr"


def parse_cdhit_clstr(clstr_path, prefix=""):
    """Parses a CD-HIT .clstr file into {protein_id: cluster_id}."""
    cluster_map = {}
    current_cluster = None
    with open(clstr_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">Cluster"):
                current_cluster = f"{prefix}{line.split()[1]}"
            elif line:
                protein_id = line.split(">")[1].split("...")[0]
                cluster_map[protein_id] = current_cluster
    return cluster_map


def cluster_full_pool(all_sequences_df, output_dir, identity=0.4, word_size=2, memory_mb=4000, threads=4):
    """
    Runs CD-HIT ONCE on the full universe of unique host/pathogen sequences
    (positives + all negative pools combined), producing the master ID maps
    and cluster maps that every downstream ratio-dataset will reuse.

    Returns: host_id_map_df, pathogen_id_map_df, host_cluster_map, pathogen_cluster_map
    """
    os.makedirs(output_dir, exist_ok=True)

    host_map_df, pathogen_map_df, host_seqs, pathogen_seqs = build_id_maps(all_sequences_df)
    host_map_df.to_csv(f"{output_dir}/host_id_map.csv", index=False)
    pathogen_map_df.to_csv(f"{output_dir}/pathogen_id_map.csv", index=False)

    host_fasta = f"{output_dir}/host_proteins.fasta"
    pathogen_fasta = f"{output_dir}/pathogen_proteins.fasta"
    write_fasta(host_seqs, host_fasta, "HOST")
    write_fasta(pathogen_seqs, pathogen_fasta, "PATHOGEN")

    host_clstr = run_cdhit(host_fasta, f"{output_dir}/host_cdhit", identity, word_size, memory_mb, threads)
    pathogen_clstr = run_cdhit(pathogen_fasta, f"{output_dir}/pathogen_cdhit", identity, word_size, memory_mb, threads)

    host_cluster_map = parse_cdhit_clstr(host_clstr, prefix="hostclust_")
    pathogen_cluster_map = parse_cdhit_clstr(pathogen_clstr, prefix="pathclust_")

    print(f"{len(host_cluster_map)} host proteins -> {len(set(host_cluster_map.values()))} clusters")
    print(f"{len(pathogen_cluster_map)} pathogen proteins -> {len(set(pathogen_cluster_map.values()))} clusters")

    with open(f"{output_dir}/host_cluster_map.json", "w") as f:
        json.dump(host_cluster_map, f)
    with open(f"{output_dir}/pathogen_cluster_map.json", "w") as f:
        json.dump(pathogen_cluster_map, f)

    return host_map_df, pathogen_map_df, host_cluster_map, pathogen_cluster_map


def apply_cluster_maps(df, host_map_df, pathogen_map_df, host_cluster_map, pathogen_cluster_map):
    """
    Merges protein IDs and cluster assignments onto any dataframe (positive
    pairs, a negative pool, or a full ratio-built dataset) using the master
    maps produced by cluster_full_pool(). Safe to call repeatedly on
    different subsets, since the maps never change.
    """
    df = df.merge(host_map_df, on="host_sequence", how="left")
    df = df.merge(pathogen_map_df, on="pathogen_sequence", how="left")

    df["host_cluster"] = df["host_protein_id"].map(host_cluster_map)
    df["pathogen_cluster"] = df["pathogen_protein_id"].map(pathogen_cluster_map)

    assert df["host_protein_id"].isna().sum() == 0, "Unmapped host sequence found!"
    assert df["pathogen_protein_id"].isna().sum() == 0, "Unmapped pathogen sequence found!"
    assert df["host_cluster"].isna().sum() == 0, "Unmapped host cluster found!"
    assert df["pathogen_cluster"].isna().sum() == 0, "Unmapped pathogen cluster found!"

    return df

Driver

In [ ]:
!apt-get update -qq
!apt-get install -y cd-hit

In [ ]:
cdhit_root = f"{file_path}/CDHIT"

# Combine ALL validated data (positives + every species' full negative pool)
# to build the complete universe of sequences CD-HIT should see.
all_sequences_for_clustering = pd.concat(
    list(validated_positive.values()) + list(validated_negative.values()),
    ignore_index=True,
)

host_map_df, pathogen_map_df, host_cluster_map, pathogen_cluster_map = cluster_full_pool(
    all_sequences_for_clustering,
    output_dir=f"{cdhit_root}/master_clustering",
)

# Apply the master cluster maps onto each validated positive/negative dataframe
for species in validated_positive:
    validated_positive[species] = apply_cluster_maps(
        validated_positive[species], host_map_df, pathogen_map_df, host_cluster_map, pathogen_cluster_map
    )
    validated_negative[species] = apply_cluster_maps(
        validated_negative[species], host_map_df, pathogen_map_df, host_cluster_map, pathogen_cluster_map
    )

print("Cluster assignments applied to all validated positive and negative dataframes.")

# Module 3: Balanced and Imbalanced Data Construction

This module creates the final datasets with different **negative-to-positive ratios** by randomly sampling from the species-specific validated negative pools generated in the previous modules. A balanced dataset is constructed by selecting an equal number of negative interactions relative to positive interactions, while imbalanced datasets are generated using predefined higher numbers of negative interactions. This allows model performance to be investigated under different class-distribution scenarios

##**Workflow**

```text
Validated positive data + Negative pools
                  ↓
       Species-specific sampling
                  ↓
        ┌─────────┼─────────┐
        ↓         ↓         ↓
       1:1       1:5       1:10
        ↓         ↓         ↓
       Merge positive and negative pairs
                  ↓
              Shuffle
                  ↓
          Final ML datasets
```

For each pathogen species, the number of negative pairs sampled is determined from the number of positive pairs and the selected ratio.

For example:

* **1:1** → one negative pair per positive pair
* **1:5** → five negative pairs per positive pair
* **1:10** → ten negative pairs per positive pair

Sampling is performed independently for each species using a fixed random seed to ensure reproducibility.

After sampling, all positive and negative pairs are combined and assigned their corresponding labels:

* `label = 1` → positive interaction
* `label = 0` → negative pair

The combined dataset is then randomly shuffled before being saved.

## Cluster Information

Because clustering was completed in **Module 2**, each pair already contains:

* `host_cluster`
* `pathogen_cluster`

The module verifies that these columns are present and contain no missing values. This ensures that the master cluster assignments are preserved across all ratio-specific datasets.

## Output Datasets

The module generates three datasets:

```text
merged_balanced_hpi_dataset.csv   → 1:1
merged_R5_hpi_dataset.csv         → 1:5
merged_R10_hpi_dataset.csv        → 1:10
```

These datasets are used as inputs for the subsequent **data partitioning and machine-learning modules**.


In [ ]:

def build_imbalanced_dataset(
    validated_positive_dict,
    validated_negative_dict,
    positive_counts,
    ratio,
    output_path,
    random_state=42,
):
    negative_samples = []
    for species, n_pos in positive_counts.items():
        n_neg = n_pos * ratio
        pool = validated_negative_dict[species]
        replace = len(pool) < n_neg
        sample = pool.sample(n=n_neg, random_state=random_state, replace=replace)
        negative_samples.append(sample)
        print(f"[{species}] sampled {n_neg} negative pairs (pool size {len(pool)}, replace={replace})")

    df_neg = pd.concat(negative_samples, ignore_index=True)
    df_pos = pd.concat(validated_positive_dict.values(), ignore_index=True)

    df_pos = df_pos.copy()
    df_neg = df_neg.copy()
    df_pos["label"] = 1
    df_neg["label"] = 0

    df_combined = pd.concat([df_pos, df_neg], ignore_index=True)
    df_combined = df_combined.sample(frac=1, random_state=random_state).reset_index(drop=True)

    # sanity check: cluster columns should already be present, no re-clustering needed
    assert "host_cluster" in df_combined.columns
    assert "pathogen_cluster" in df_combined.columns
    assert df_combined["host_cluster"].isna().sum() == 0
    assert df_combined["pathogen_cluster"].isna().sum() == 0

    df_combined.to_csv(output_path, index=False)
    print(f"Saved {len(df_combined)} rows to {output_path} "
          f"(positive={len(df_pos)}, negative={len(df_neg)})")
    return df_combined

### Reload validated positive/negative data (if starting a new session)

If `validated_positive` / `validated_negative` are no longer in memory (e.g.,
after a kernel restart), reload them from disk before continuing.

Note: clustering (Module 2) was run on the *combined* validated data, and its
`host_cluster` / `pathogen_cluster` columns were merged back in-memory at that
time — they are **not** saved in the raw `validated_positive_*.csv` /
`validated_negative_*.csv` files. Reloading from these files alone loses the
cluster assignments, so the saved cluster maps must be re-applied afterward
using `apply_cluster_maps()` before proceeding to Module 4.

In [ ]:
import pandas as pd

file_path = '/content/my_drive/MyDrive/HPI'
species_list = ["bacillus", "ecoli", "francisella", "yersinia"]

validated_positive = {
    species: pd.read_csv(f"{file_path}/validated_positive_{species}.csv")
    for species in species_list
}

validated_negative = {
    species: pd.read_csv(f"{file_path}/validated_negative_{species}.csv")
    for species in species_list
}

import json

cdhit_root = f"{file_path}/CDHIT"

with open(f"{cdhit_root}/master_clustering/host_cluster_map.json") as f:
    host_cluster_map = json.load(f)
with open(f"{cdhit_root}/master_clustering/pathogen_cluster_map.json") as f:
    pathogen_cluster_map = json.load(f)

host_map_df = pd.read_csv(f"{cdhit_root}/master_clustering/host_id_map.csv")
pathogen_map_df = pd.read_csv(f"{cdhit_root}/master_clustering/pathogen_id_map.csv")

for species in species_list:
    validated_positive[species] = apply_cluster_maps(
        validated_positive[species], host_map_df, pathogen_map_df, host_cluster_map, pathogen_cluster_map
    )
    validated_negative[species] = apply_cluster_maps(
        validated_negative[species], host_map_df, pathogen_map_df, host_cluster_map, pathogen_cluster_map
    )

for species in species_list:
    assert "host_cluster" in validated_positive[species].columns
    assert "host_cluster" in validated_negative[species].columns

print("Reload complete — cluster assignments reapplied to all species.")

##Build the Different Ratio Datasets

In [ ]:

positive_counts = {species: len(df) for species, df in validated_positive.items()}
print("Positive counts per species:", positive_counts)

ratio_config = {
    1:  f"{file_path}/merged_balanced_hpi_dataset.csv",
    5:  f"{file_path}/merged_R5_hpi_dataset.csv",
    10: f"{file_path}/merged_R10_hpi_dataset.csv",
}

merged_datasets = {}
for ratio, out_path in ratio_config.items():
    merged_datasets[ratio] = build_imbalanced_dataset(
        validated_positive_dict=validated_positive,
        validated_negative_dict=validated_negative,
        positive_counts=positive_counts,
        ratio=ratio,
        output_path=out_path,
    )

# Module 4: Homology Aware Data Partioning
This module performs a stringent partitioning strategy that prevents both host and pathogen homology clusters from being shared across train/test partitions or cross-validation folds.

Host and pathogen clusters are connected using a Union-Find approach. Clusters that co-occur in an interaction are grouped together, and these combined groups are then assigned to partitions.

This ensures that Host homology clusters do not span partitions and
Pathogen homology clusters do not span partitions.

**This strategy is used as an independent generalization check. The primary model development uses host-only homology-aware partitioning.**

In [ ]:

import os
import pandas as pd


class UnionFind:
    def __init__(self):
        self.parent = {}

    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]  # path compression
            x = self.parent[x]
        return x

    def union(self, x, y):
        root_x, root_y = self.find(x), self.find(y)
        if root_x != root_y:
            self.parent[root_x] = root_y


def build_combined_groups(df):
    """
    Builds combined host+pathogen groups via Union-Find: any host_cluster
    and pathogen_cluster that co-occur in a pair are merged into one group.
    Adds a 'combined_group' column to df.
    """
    uf = UnionFind()
    for h, p in zip(df["host_cluster"], df["pathogen_cluster"]):
        uf.union(h, p)

    df = df.copy()
    df["combined_group"] = df["host_cluster"].apply(uf.find)

    n_groups = df["combined_group"].nunique()
    group_sizes = df["combined_group"].value_counts()
    print(f"{n_groups} combined groups formed from host+pathogen clusters")
    print(group_sizes.describe())

    return df


def dual_homology_split(df, output_dir, test_size=0.2, seed=42):
    """
    Splits df into train/test using combined (host+pathogen) group disjointness.
    Verifies zero leakage on host cluster, pathogen cluster, host protein,
    and pathogen protein.
    """
    os.makedirs(output_dir, exist_ok=True)

    df = build_combined_groups(df)

    rng = np.random.default_rng(seed)
    groups = df["combined_group"].unique()
    rng.shuffle(groups)
    n_test = int(len(groups) * test_size)
    test_groups = set(groups[:n_test])

    df["partition"] = df["combined_group"].apply(lambda g: "test" if g in test_groups else "train")

    train_df = df[df["partition"] == "train"].drop(columns=["partition"])
    test_df = df[df["partition"] == "test"].drop(columns=["partition"])

    dropped_note = (
        f"Train: {len(train_df)} ({len(train_df)/len(df):.1%}) | "
        f"Test: {len(test_df)} ({len(test_df)/len(df):.1%})"
    )
    print(dropped_note)
    print("Train class balance:\n", train_df["label"].value_counts(normalize=True))
    print("Test class balance:\n", test_df["label"].value_counts(normalize=True))

    host_overlap = set(train_df["host_protein_id"]) & set(test_df["host_protein_id"])
    path_overlap = set(train_df["pathogen_protein_id"]) & set(test_df["pathogen_protein_id"])
    host_cluster_overlap = set(train_df["host_cluster"]) & set(test_df["host_cluster"])
    path_cluster_overlap = set(train_df["pathogen_cluster"]) & set(test_df["pathogen_cluster"])

    print(f"Host protein overlap: {len(host_overlap)} | Pathogen protein overlap: {len(path_overlap)}")
    print(f"Host cluster overlap: {len(host_cluster_overlap)} | Pathogen cluster overlap: {len(path_cluster_overlap)}")

    train_df.to_csv(f"{output_dir}/dual_homology_train_split.csv", index=False)
    test_df.to_csv(f"{output_dir}/dual_homology_test_split.csv", index=False)

    return train_df, test_df


def dual_homology_cv(train_df, output_dir, n_splits=10):
    """
    10-fold CV using combined (host+pathogen) group disjointness. Included
    for completeness/comparison — expect uneven fold sizes given the
    heterogeneous group-size issue documented above.
    """
    from sklearn.model_selection import GroupKFold

    fold_dir = f"{output_dir}/cv_folds_dual_aware"
    os.makedirs(fold_dir, exist_ok=True)

    train_df = build_combined_groups(train_df)

    gkf = GroupKFold(n_splits=n_splits)
    fold_summary = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, groups=train_df["combined_group"])):
        fold_train = train_df.iloc[train_idx].reset_index(drop=True)
        fold_val = train_df.iloc[val_idx].reset_index(drop=True)

        host_cluster_overlap = set(fold_train["host_cluster"]) & set(fold_val["host_cluster"])
        path_cluster_overlap = set(fold_train["pathogen_cluster"]) & set(fold_val["pathogen_cluster"])
        host_protein_overlap = set(fold_train["host_protein_id"]) & set(fold_val["host_protein_id"])
        path_protein_overlap = set(fold_train["pathogen_protein_id"]) & set(fold_val["pathogen_protein_id"])

        fold_train.to_csv(f"{fold_dir}/fold{fold}_train.csv", index=False)
        fold_val.to_csv(f"{fold_dir}/fold{fold}_val.csv", index=False)

        fold_summary.append({
            "fold": fold,
            "n_train": len(fold_train),
            "n_val": len(fold_val),
            "val_pos_frac": fold_val["label"].mean(),
            "host_cluster_overlap": len(host_cluster_overlap),
            "pathogen_cluster_overlap": len(path_cluster_overlap),
            "host_protein_overlap": len(host_protein_overlap),
            "pathogen_protein_overlap": len(path_protein_overlap),
        })

    fold_summary_df = pd.DataFrame(fold_summary)
    fold_summary_df.to_csv(f"{fold_dir}/fold_summary.csv", index=False)
    print(fold_summary_df)
    return fold_summary_df

##Run Host-Pathogen Dual Aware Partitioning

In [ ]:
import numpy as np

ratio_dirs = {1: f"{cdhit_root}/balanced_CDHIT", 5: f"{cdhit_root}/R5_CDHIT", 10: f"{cdhit_root}/R10_CDHIT"}

dual_splits = {}
for ratio, df in merged_datasets.items():
    print(f"\n=== Ratio 1:{ratio} — Dual Homology-Aware Split ===")
    dual_splits[ratio] = dual_homology_split(df, output_dir=ratio_dirs[ratio])

# Optional: dual-aware CV, run only if you want this stringent comparison
dual_cv_summaries = {}
for ratio, (train_df, test_df) in dual_splits.items():
    print(f"\n=== Ratio 1:{ratio} — Dual Homology-Aware CV ===")
    dual_cv_summaries[ratio] = dual_homology_cv(train_df, output_dir=ratio_dirs[ratio])

##**NOTE**
Note: The dual-sided (host + pathogen) homology-aware partitioning strategy was evaluated but not adopted for downstream analysis. Merging host and pathogen homology clusters that co-occur in interaction pairs produced highly imbalanced combined groups, with a single dominant group accounting for 76%, 99%, and over 99.9% of the dataset at the 1:1, 1:5, and 1:10 imbalance ratios respectively. This resulted in impractical, and in some cases fully inverted, train/test splits and cross-validation folds (e.g., a test partition larger than the training partition, or a fold with no positive examples). Consequently, host-only disjoint partitioning was adopted as the primary strategy for all subsequent train/test splitting, cross-validation, and LOPO evaluation, with the dual-homology results retained here only as a documented robustness check illustrating this limitation.

This outcome is as a result of the relatively small and highly cross-linked dataset used here; for datasets with a larger and more diverse pool of host-pathogen interactions, dual-sided partitioning may produce well-balanced groups and, where feasible, should be preferred over host-only partitioning for a more rigorous leakage-free evaluation.



# Module 5: Host Only Homology Aware Data Partioining
Host protein homology clusters are used to guide the train–test partitioning and cross-validation procedure, ensuring that homologous host proteins are not distributed across different data partitions. This strategy was adopted to retain sufficient data for model development while reducing the potential for sequence information leakage

##5a: Train/Test Split
This module divides the dataset into training and testing sets while maintaining the required class distribution.


In [ ]:

import numpy as np


def assign_host_partitions(df, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)
    host_clusters = df["host_cluster"].unique()
    rng.shuffle(host_clusters)
    n_test = int(len(host_clusters) * test_size)
    test_clusters = set(host_clusters[:n_test])
    return {c: ("test" if c in test_clusters else "train") for c in host_clusters}


def host_disjoint_split(df, output_dir, test_size=0.2, seed=42):
    """
    Splits df into host-disjoint train/test partitions, verifies no host
    leakage, and saves both partitions plus a leakage report to output_dir.
    """
    os.makedirs(output_dir, exist_ok=True)

    partition = assign_host_partitions(df, test_size, seed)
    df = df.copy()
    df["partition"] = df["host_cluster"].map(partition)

    train_df = df[df["partition"] == "train"].drop(columns=["partition"])
    test_df = df[df["partition"] == "test"].drop(columns=["partition"])

    host_overlap = set(train_df["host_protein_id"]) & set(test_df["host_protein_id"])
    host_cluster_overlap = set(train_df["host_cluster"]) & set(test_df["host_cluster"])
    path_overlap = set(train_df["pathogen_protein_id"]) & set(test_df["pathogen_protein_id"])

    print(f"Train: {len(train_df)} ({len(train_df)/len(df):.1%}) | Test: {len(test_df)} ({len(test_df)/len(df):.1%})")
    print(f"Host protein overlap: {len(host_overlap)} | Host cluster overlap: {len(host_cluster_overlap)}")
    print(f"Pathogen protein overlap: {len(path_overlap)} "
          f"of {df['pathogen_protein_id'].nunique()} unique pathogen proteins")

    train_df.to_csv(f"{output_dir}/train_split.csv", index=False)
    test_df.to_csv(f"{output_dir}/test_split.csv", index=False)
    train_df.to_pickle(f"{output_dir}/train_split.pkl")
    test_df.to_pickle(f"{output_dir}/test_split.pkl")

    return train_df, test_df

In [ ]:
splits = {}
for ratio, df in merged_datasets.items():
    print(f"\n--- Ratio 1:{ratio} ---")
    splits[ratio] = host_disjoint_split(df, output_dir=ratio_dirs[ratio])

##5b: Host-Disjoint 10-Fold Cross-Validation
This module splits the training dataset into 10 host-disjoint folds for cross-validation. The resulting folds are then used for hyperparameter tuning and model selection

In [ ]:


from sklearn.model_selection import GroupKFold


def host_disjoint_cv(train_df, output_dir, n_splits=10):
    """
    Builds host-cluster-disjoint CV folds from a training partition, saves
    each fold, and reports leakage/balance diagnostics per fold.
    """
    fold_dir = f"{output_dir}/cv_folds"
    os.makedirs(fold_dir, exist_ok=True)

    gkf = GroupKFold(n_splits=n_splits)
    fold_summary = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, groups=train_df["host_cluster"])):
        fold_train = train_df.iloc[train_idx].reset_index(drop=True)
        fold_val = train_df.iloc[val_idx].reset_index(drop=True)

        cluster_overlap = set(fold_train["host_cluster"]) & set(fold_val["host_cluster"])
        protein_overlap = set(fold_train["host_protein_id"]) & set(fold_val["host_protein_id"])

        fold_train.to_csv(f"{fold_dir}/fold{fold}_train.csv", index=False)
        fold_val.to_csv(f"{fold_dir}/fold{fold}_val.csv", index=False)

        fold_summary.append({
            "fold": fold,
            "n_train": len(fold_train),
            "n_val": len(fold_val),
            "val_pos_frac": fold_val["label"].mean(),
            "host_cluster_overlap": len(cluster_overlap),
            "host_protein_overlap": len(protein_overlap),
        })

    fold_summary_df = pd.DataFrame(fold_summary)
    fold_summary_df.to_csv(f"{fold_dir}/fold_summary.csv", index=False)
    print(fold_summary_df)
    return fold_summary_df

In [ ]:
cv_summaries = {}
for ratio, (train_df, test_df) in splits.items():
    print(f"\n--- Ratio 1:{ratio} CV ---")
    cv_summaries[ratio] = host_disjoint_cv(train_df, output_dir=ratio_dirs[ratio])

##5c: Leave One Pathogen Out
This module creates Leave-One-Pathogen-Out (LOPO) partitions by holding out one pathogen species at a time.

For each iteration, one pathogen is assigned to the test set, while the remaining pathogens form the training set. The resulting partitions are used later to evaluate the model's ability to generalize to an unseen pathogen.

In [ ]:
def lopo_host_disjoint_folds(df, output_dir, species_col="pathogen_specie"):
    """
    Builds host-disjoint LOPO folds: for each species, holds it out as the
    test set, and removes from training any pair whose host cluster also
    appears in that species' test set.
    """
    fold_dir = f"{output_dir}/lopo_folds_host_disjoint"
    os.makedirs(fold_dir, exist_ok=True)

    species_list = df[species_col].unique()
    summary = []

    for held_out in species_list:
        test_fold = df[df[species_col] == held_out].copy()
        train_candidates = df[df[species_col] != held_out].copy()

        test_clusters = set(test_fold["host_cluster"])
        train_fold = train_candidates[~train_candidates["host_cluster"].isin(test_clusters)].copy()
        dropped = len(train_candidates) - len(train_fold)

        safe_name = str(held_out).replace(" ", "_")
        train_fold.to_csv(f"{fold_dir}/lopo_{safe_name}_train.csv", index=False)
        test_fold.to_csv(f"{fold_dir}/lopo_{safe_name}_test.csv", index=False)

        cluster_overlap = set(train_fold["host_cluster"]) & set(test_fold["host_cluster"])
        protein_overlap = set(train_fold["host_protein_id"]) & set(test_fold["host_protein_id"])

        summary.append({
            "held_out_species": held_out,
            "n_train": len(train_fold),
            "n_test": len(test_fold),
            "dropped_pairs": dropped,
            "pct_dropped": dropped / len(train_candidates),
            "test_pos_frac": test_fold["label"].mean(),
            "host_cluster_overlap": len(cluster_overlap),
            "host_protein_overlap": len(protein_overlap),
        })

    summary_df = pd.DataFrame(summary)
    summary_df.to_csv(f"{fold_dir}/lopo_summary.csv", index=False)
    print(summary_df)
    return summary_df

### Run LOPO

In [ ]:
lopo_summaries = {}
for ratio, df in merged_datasets.items():
    print(f"\n--- Ratio 1:{ratio} LOPO ---")
    lopo_summaries[ratio] = lopo_host_disjoint_folds(df, output_dir=ratio_dirs[ratio])

#PARTITIONING COMPLETE

At the end of these notebooks, all required data partitions are prepared for the next stage of the workflow: **Feature Extraction**.

The resulting train/test, cross-validation, and LOPO partitions are ready for sequence feature extraction and subsequent machine-learning analysis.
